In [2]:
# 1. Standard Imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from dbconfig import engine

print("Environment Ready.")

Environment Ready.


In [74]:
%%sql
with total_matches as
(select team1 as team, count(distinct match_id) as c
from ipl_master_view
group by team1

union all

select team2 as team, count(distinct match_id) as c
from ipl_master_view
group by team2)

select team, sum(c) as match_count
from total_matches
group by team
order by match_count desc;



,team,match_count
0,Mumbai Indians,261
1,Sunrisers Hyderabad,257
2,Royal Challengers Bengaluru,255
3,Delhi Capitals,252
4,Kolkata Knight Riders,251
5,Punjab Kings,246
6,Chennai Super Kings,238
7,Rajasthan Royals,221
8,Pune Warriors,46
9,Gujarat Titans,45


In [75]:
%%sql
select winner, count(distinct match_id) as win_count
from ipl_master_view
where winner is not null
group by winner
order by win_count desc;

,winner,win_count
0,Mumbai Indians,144
1,Chennai Super Kings,138
2,Kolkata Knight Riders,131
3,Royal Challengers Bengaluru,123
4,Sunrisers Hyderabad,117
5,Delhi Capitals,115
6,Punjab Kings,112
7,Rajasthan Royals,112
8,Gujarat Titans,28
9,Lucknow Super Giants,24


In [57]:
%%sql
select distinct result
from ipl_master_view;

,result
0,no result
1,runs
2,tie
3,wickets


In [63]:
%%sql
with runs_margin as (
select avg(result_margin) as r_m
from ipl_master_view
where result = 'runs'
and winner = 'Mumbai Indians'),

wickets_margin as (
select avg(result_margin) as w_m
from ipl_master_view
where result = 'wickets'
and winner = 'Mumbai Indians')

select r_m as runs_margin, w_m as wickets_margin
from runs_margin, wickets_margin;

,runs_margin,wickets_margin
0,32.241587,6.131718


In [84]:
result = match_count.merge(win_count, left_on='team', right_on='winner', how='left')
result = result.drop(columns=['winner'])
result['win_pct'] = (result['win_count'] / result['match_count'] * 100.0).round(2)

In [85]:
win_pct_top_3_played = result[result['team'].isin(['Mumbai Indians', 'Chennai Super Kings', 'Royal Challengers Bengaluru'])].sort_values(by = 'win_pct', ascending = False)
win_pct_top_3_played

,team,match_count,win_count,win_pct
6,Chennai Super Kings,238,138,57.98
0,Mumbai Indians,261,144,55.17
2,Royal Challengers Bengaluru,255,123,48.24
